# **I. IMPORTING THE NECESSARY LIBRARIES**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, precision_recall_curve, auc, roc_auc_score, average_precision_score

****

# **II. LOADING THE DATAFRAME**

In [ ]:
# Initialize the original dataframe
df = pd.read_csv("/kaggle/input/datasets/margielynatienza/music-genre/music_genre.csv")

if df is not None: #Checks if the dataframe is loaded
    print("Loaded: Music Genre") 
    print(f"Rows: {len(df)} || Columns: {len(df.columns)}")

****

# **III. ANALYZING THE ORIGINAL DATAFRAME**

In [ ]:
df #check original dataset

In [ ]:
df.columns #print all column names

In [ ]:
df.info() #print summary of information

In [ ]:
df.describe() #generates description

In [ ]:
df.shape #prints number of rows and columns

In [ ]:
df.head(5) #prints the first 5 rows

In [ ]:
df.tail(5) #prints the last 5 rows

****

# **IV. DUPLICATING THE DATAFRAME**

## **V.I. DUPLICATING THE DATA SET**

In [ ]:
data = df.copy() #independent copy of original dataset for data manipulation

In [ ]:
data #print the duplicated dataset

# **V. DATA CLEANING**

## **V.I. ERASING DUPLICATE ROWS**

In [ ]:
data.duplicated() #check for duplicate records

In [ ]:
data.duplicated().sum() #gets the amount of duplicated rows

In [ ]:
data = data.drop_duplicates() #deletes all the duplicated rows

In [ ]:
data #check the dataset for differences

In [ ]:
data.duplicated().sum() #confirming no more duplicates

## **V.II. CHECKING FOR MISSING VALUES**

In [ ]:
data.count() #checks name of each column and amount of non-missing entries

In [ ]:
data.isnull().sum() #checks number of missing values in each column

In [ ]:
data["tempo"].value_counts() # checks for missing placeholders

In [ ]:
data = data.replace("?", np.nan) # removes the unknown placeholder:'?' 

In [ ]:
data["tempo"].value_counts() # checks for missing placeholders

## **V.III. REMOVE IRRELEVANT COLUMNS**

In [ ]:
irrelevant_columns = data[["instance_id", "artist_name", "track_name", "obtained_date"]] #create array of columns to remove

#loop to remove columns
for i in irrelevant_columns:
    data = data.drop(i, axis = "columns")

In [ ]:
data #check data to see

In [ ]:
data.shape # check shape of data

## **V.IV. CHECKING IMPOSSIBLE VALUES**

In [ ]:
data.loc[data["duration_ms"] <= 0, "duration_ms"] = np.nan # replace all the values less than or equal to 0 in the 'duration_ms' column with NaN

## **V.V. FIXING INCONSISTENT FORMATS**

In [ ]:
data["tempo"] = pd.to_numeric(data["tempo"]) # converts string to float

In [ ]:
rounded_columns = data[["acousticness", "danceability", "duration_ms", "energy", "instrumentalness", "liveness", "speechiness", "tempo"]] #create array of columns

#loop to round values in columns
for i in rounded_columns:
    data[i] = data[i].round(3)

data.head()

## **V.VI. CHECKING INVALID LABELS**

In [ ]:
data["music_genre"].isnull().sum() # check target label for missing entries

In [ ]:
data = data.dropna(subset = ["music_genre"]) # drops all rows with missing values based on the target variable

In [ ]:
data["music_genre"].isnull().sum() # confirm if there are no missing entries in the target variable

In [ ]:
data["music_genre"].value_counts()

In [ ]:
le = LabelEncoder()

In [ ]:
data["music_genre"].value_counts()

In [ ]:
data["music_genre"] = le.fit_transform(data["music_genre"])

## **V.VII. RE-ANALYZING THE DATA**

In [ ]:
data.shape # check shape of dataset post data cleaning

In [ ]:
data # check dataset post data cleaning

****

# **VI. DATA SPLITTING**

## **VI.I. FEATURE LABEL SEPARATION**

In [ ]:
X = data.drop("music_genre", axis = "columns") #stores a version of the data without the target variable as 'X'

In [ ]:
y = data["music_genre"] #stores the target variable as 'y'

In [ ]:
X.head() #displays X's first 5 rows

In [ ]:
y.head() #displays y's first 5 rows

## **VI.II. TRAIN TEST SPLIT**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #splits X and Y into training and testing data

## **VI.III. ANALYZING SPLITTED RESULTS**

In [ ]:
data.shape #check the shape of the entire dataset

In [ ]:
X_train.shape #check the shape of the training features

In [ ]:
X_test.shape #check the shape of the testing features

In [ ]:
y_train.shape #check the shape of the training target variable

In [ ]:
y_test.shape #check the shape of the testing target variable

In [ ]:
print("Training Music Qualities:", len(X_train)) #check amount of rows of training features
print("Training Music Genres:", len(y_train)) #check amount of rows of the training target variable

print("Test Music Qualities:", len(X_test)) #check amount of rows of testing features
print("Test Music Genres:", len(y_test)) #check amount of rows of the testing target variable

****

# **VII. DATA PREPROCESSING**

In [ ]:
# create copies so original training data remain unchanged
X_train_prepared = X_train.copy()
X_test_prepared = X_test.copy()

## **VII.I. HANDLING MISSING VALUES**

In [ ]:
X_train_prepared.isnull().sum() #check missing values in training data

In [ ]:
X_test_prepared.isnull().sum() #check missing values in testing data

In [ ]:
# Mean Imputation on 'duration_ms' column
duration_mean = X_train_prepared["duration_ms"].mean()
X_train_prepared["duration_ms"] = X_train_prepared["duration_ms"].fillna(duration_mean) 
X_test_prepared["duration_ms"] = X_test_prepared["duration_ms"].fillna(duration_mean)

In [ ]:
# Mean Imputation on 'tempo' column
tempo_mean = X_train_prepared["tempo"].mean()
X_train_prepared["tempo"] = X_train_prepared["tempo"].fillna(tempo_mean) 
X_test_prepared["tempo"] = X_test_prepared["tempo"].fillna(tempo_mean)

In [ ]:
X_train_prepared.isnull().sum() # confirm filled values in training data

In [ ]:
X_test_prepared.isnull().sum() # confirm filled values in testing data

## **VII.II. FEATURE SCALING**

In [ ]:
numeric_columns = ["popularity", "acousticness", "danceability", "duration_ms", "energy", "instrumentalness","liveness", "loudness", "speechiness", "tempo", "valence"] # All the columns you want to scale

In [ ]:
scaler = StandardScaler() # create scaler object

In [ ]:
X_train_prepared[numeric_columns] = scaler.fit_transform(X_train_prepared[numeric_columns]) #fit scaler on training data & transform training data

In [ ]:
X_test_prepared[numeric_columns] = scaler.transform(X_test_prepared[numeric_columns]) # apply same scaling rule on testing data

In [ ]:
X_train_prepared.head() # check the post-scaled training data

## **VII.III. CATEGORICAL ENCODING**

In [ ]:
X_train_prepared["mode"] # check training values in 'mode' column

In [ ]:
# converts into binary coding for training data
X_train_prepared["mode"] = X_train_prepared["mode"].map({
    "Major": 1,
    "Minor": 0
})

In [ ]:
X_train_prepared["mode"] # confirm training values were changed

In [ ]:
X_test_prepared["mode"] # check testing values in the 'mode' column

In [ ]:
# converts into binary coding for testing data
X_test_prepared["mode"] = X_test_prepared["mode"].map({
    "Major": 1,
    "Minor": 0
})

In [ ]:
X_test_prepared["mode"] # confirm testing values were changed in the 'mode' column

In [ ]:
# converts 'key' column into numerical columns using one-hot encoding
X_train_prepared = pd.get_dummies(X_train_prepared, columns = ["key"], dtype = int)
X_test_prepared = pd.get_dummies(X_test_prepared, columns = ["key"], dtype = int)

In [ ]:
X_train_prepared.mean() # get the mean of each column

In [ ]:
X_train_prepared.std() # get the standard deviation of each column

In [ ]:
X_test_prepared.head() # check the post-scaled testing data

## **VII.IV. HANDLING IMBALANCE**

In [ ]:
data["music_genre"].value_counts(normalize=True) # counts all the given values

In [ ]:
data["music_genre"].value_counts().plot.pie(autopct="%.2f") # create pie chart of the counted values

****

# **VIII. MODEL TRAINING**

## **VIII.I. MODEL SELECTION**

In [ ]:
clf = OneVsRestClassifier(LogisticRegression())  # create Logistic Regression model along with a multiclass strategy 'OneVsRestClassifier'

In [ ]:
clf # display selected model

In [ ]:
X_train_prepared.head() # training features

In [ ]:
y_train.head() # training label

## **VIII.II. TRAINING THE MODEL**

In [ ]:
clf.fit(X_train_prepared, y_train) # train the Logistic Regression model

****

# **IX. MODEL EVALUATION**

## **IX.I. MODEL TESTING**

In [ ]:
y_pred = clf.predict(X_test_prepared) # predicts values

In [ ]:
pred_prob = clf.predict_proba(X_test_prepared) # predicts class probabilities

In [ ]:
pred_prob[:5] # display the first 5 class probabilities

In [ ]:
y_binarized = label_binarize(y_test, classes=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) # binarizes y values

In [ ]:
y_pred[:10] # display predicted values

In [ ]:
y_pred.shape # display shape of predicted values

## **IX.II. COMPARING THE RESULTS**

In [ ]:
# comapring the actual values and predicted values
comparison = pd.DataFrame({
    "Original row": y_test.index,
    "Actual music genre": y_test.to_numpy(),
    "Predicted music genre": y_pred
})

display(comparison.head(10)) # display the comparison, showing 10 rows, and rounded off to two decimal points

## **IX.III. PERFORMANCE EVALUATION METRICS**

In [ ]:
# initialize all the performance evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average=None)
recall = recall_score(y_test, y_pred, average=None)
f1 = f1_score(y_test, y_pred, average=None)

In [ ]:
# display precision score alongside genre
precision_eval = pd.DataFrame({
    "Genre" : ["Electronic", "Anime", "Jazz", "Alternative", "Country", "Rap", "Blues", "Rock", "Classical", "Hip-Hop"],
    "Precision Score" : precision
})

precision_eval.round(2) # rounds off values into two decimal points

In [ ]:
# display recall score alongside genre
recall_eval = pd.DataFrame({
    "Genre" : ["Electronic", "Anime", "Jazz", "Alternative", "Country", "Rap", "Blues", "Rock", "Classical", "Hip-Hop"],
    "Recall Score" : recall
})

recall_eval.round(2) # rounds off values into two decimal points

In [ ]:
# display f1 score alongside genre
f1_eval = pd.DataFrame({
    "Genre" : ["Electronic", "Anime", "Jazz", "Alternative", "Country", "Rap", "Blues", "Rock", "Classical", "Hip-Hop"],
    "F1 Score" : f1
})

f1_eval.round(2) # rounds off values into two decimal points

In [ ]:
# creates a table for evaluating each score
evaluation = pd.DataFrame({
    "Metric": ["Accuracy Score", "Precision Score", "Recall Score", "F1 Score"],
    "Average Score": [accuracy, precision.mean(), recall.mean(), f1.mean()]
})

evaluation.round(2) # rounds off values into two decimal points

****

# **X. VISUALIZATION**

## **X.I. CONFUSION MATRIX DISPLAY**

In [ ]:
# create confusion matrix 
confusion = confusion_matrix(y_test, y_pred)
cm_display = ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9'])
cm_display.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix Display')
plt.show()

## **X.II. ROC CURVE**

In [ ]:
# roc curve for classes
fpr = {}
tpr = {}
thresh ={}

In [ ]:
n_class = y_binarized.shape[1] # create number of classes

In [ ]:
# plotting ROC curves based on class    
for i in range(n_class):    
    fpr[i], tpr[i], thresh[i] = roc_curve(y_test, pred_prob[:,i], pos_label=i)
    plt.plot(fpr[i], tpr[i], linestyle='--', label=f'Class {i} vs Rest')

plt.title('Multiclass ROC curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive rate')
plt.legend(loc='best')
plt.savefig('Multiclass ROC',dpi=300); 

## **X.III. PRECISION-RECALL CURVE**

In [ ]:
precision_s, recall_s, avg_p = {}, {}, {} # for precision-recall curve

In [ ]:
# plot and create precision-recall curves based on class
plt.figure(figsize=(8, 6))
for i in range(n_class):
    precision_s[i], recall_s[i], _ = precision_recall_curve(y_binarized[:, i], pred_prob[:, i])
    avg_p[i] = average_precision_score(y_binarized[:, i], pred_prob[:, i])
    auc_score = auc(recall_s[i], precision_s[i])
    plt.plot(recall_s[i], precision_s[i], lw=2, label='Class {}'.format(i))
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multi-class Precision-Recall Curve')
plt.legend()
